# Research Stack Manifold Visualizer

Interactive visualization of PIST/SVQF/FAMM quaternion manifold structures.

**Layers:**
1. PIST shell generation (integer-based coordinates)
2. Quaternion sieve (counter-rotation band-pass)
3. FAMM frustration pruning (stress tensor filtering)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sys
from pathlib import Path

sys.path.insert(0, str(Path("/home/allaun/Documents/Research Stack/5-Applications/scripts")))
from manifold_visualizer import ManifoldMapper, QuaternionField

## 1. Generate Manifold

In [ ]:
mapper = ManifoldMapper(resolution=32)
result = mapper.map_manifold()
print(f"Survivors: {result['post_famm']} / {result['total_points']}")

## 2. Visualize S³ Stereographic Projection

In [ ]:
fig = plt.figure(figsize=(12, 5))

# Raw shell points
ax1 = fig.add_subplot(121, projection='3d')
raw_pts = np.array(result['sample_points'][:50])
# Stereographic projection
x = raw_pts[:, 1] / (1 - raw_pts[:, 0] + 1e-10)
y = raw_pts[:, 2] / (1 - raw_pts[:, 0] + 1e-10)
z = raw_pts[:, 3] / (1 - raw_pts[:, 0] + 1e-10)
ax1.scatter(x, y, z, c='blue', alpha=0.6, s=20)
ax1.set_title('S³ Manifold (Stereographic)')

# Quaternion phase space
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(raw_pts[:, 0], raw_pts[:, 1], raw_pts[:, 2], c='red', alpha=0.6, s=20)
ax2.set_title('Quaternion Components (w,x,y)')

plt.tight_layout()
plt.show()

## 3. Frustration Heatmap

In [ ]:
# Simulate frustration across k-t grid
resolution = 32
frustration_map = np.zeros((resolution, resolution//2))

for k in range(resolution):
    for t in range(resolution//2):
        mass = (k+1)*(t+1)
        theta = 2 * np.pi * k / resolution
        phi = np.pi * t / resolution
        # Approximate frustration from curvature
        frustration_map[k, t] = np.abs(np.sin(theta*2) * np.cos(phi*3))

plt.figure(figsize=(10, 4))
plt.imshow(frustration_map.T, aspect='auto', cmap='hot', origin='lower')
plt.colorbar(label='Φ (frustration)')
plt.xlabel('k (shell index)')
plt.ylabel('t (time step)')
plt.title('FAMM Frustration Heatmap — Φ > 1 regions pruned')
plt.axhline(y=resolution//4, color='cyan', linestyle='--', label='φ = 1 threshold')
plt.legend()
plt.show()